# Dijkstra's Algorithm: Tiny City Simulator

Dijkstra's algorithm finds the cheapest path from one starting point to every other reachable point in a weighted graph.

Edsger Dijkstra published the algorithm in 1959 after thinking about routes between Dutch cities. Today the same cheapest-first idea appears in maps, network routing, game pathfinding, robotics, and planning tools.

In this notebook, you will build it with small objects: places, roads, a map, and a runner that explores the map.

<details>
<summary>Big idea</summary>

Keep a scoreboard of the best known cost to each place. Always explore the unsettled place with the lowest known cost next.

</details>

## 1. The Mental Model

A graph is just objects plus connections:

- **Place**: a node, like `Arcade` or `Library`
- **Road**: an edge with a cost, like travel time
- **Frontier**: places discovered but not locked in yet
- **Settled**: places whose cheapest cost is final

Dijkstra repeats one move: pick the cheapest frontier place, then try to improve its neighbors.

<details>
<summary>Hint: why the cheapest frontier place?</summary>

If all road costs are non-negative, no later route can sneak in and make that cheapest place cheaper. That is the trick that makes Dijkstra work.

</details>

## 2. Build the Objects

Implementation plan:

1. `Road` stores the destination and cost.
2. `MapGraph` stores each place and its outgoing roads.
3. `Step` records snapshots so the algorithm feels like a simulation.
4. `DijkstraRunner` owns the actual shortest-path logic.

<details>
<summary>Implementation hint</summary>

Use a priority queue for the frontier. In Python, `heapq` gives you the cheapest item first.

</details>

In [ ]:
from dataclasses import dataclass, field
from heapq import heappop, heappush
from math import inf


In [ ]:

@dataclass(frozen=True)
class Road:
    to: str
    cost: int

In [ ]:
@dataclass
class MapGraph:
    roads: dict[str, list[Road]] = field(default_factory=dict)

    def add_place(self, name: str) -> None:
        self.roads.setdefault(name, [])

    def connect(self, start: str, end: str, cost: int, two_way: bool = False) -> None:
        if cost < 0:
            raise ValueError("Dijkstra's algorithm needs non-negative road costs.")

        self.add_place(start)
        self.add_place(end)
        self.roads[start].append(Road(end, cost))

        if two_way:
            self.roads[end].append(Road(start, cost))

    def neighbors(self, place: str) -> list[Road]:
        return self.roads.get(place, [])



In [ ]:

@dataclass
class Step:
    current: str
    action: str
    distances: dict[str, float]
    frontier: list[tuple[float, str]]
    settled: set[str]



**Algorithm engine.** Define `DijkstraRunner`, the class that runs the main simulation or algorithm.


In [ ]:

class DijkstraRunner:
    def __init__(self, graph: MapGraph):
        self.graph = graph

    def shortest_paths(self, start: str) -> tuple[dict[str, float], dict[str, str | None], list[Step]]:
        if start not in self.graph.roads:
            raise ValueError(f"Unknown starting place: {start}")

        distances = {place: inf for place in self.graph.roads}
        previous = {place: None for place in self.graph.roads}
        distances[start] = 0

        frontier = [(0, start)]
        settled: set[str] = set()
        steps: list[Step] = []

        while frontier:
            current_cost, current = heappop(frontier)
            if current in settled:
                continue

            settled.add(current)
            steps.append(self._snapshot(current, "settle", distances, frontier, settled))

            for road in self.graph.neighbors(current):
                candidate = current_cost + road.cost
                if candidate < distances[road.to]:
                    distances[road.to] = candidate
                    previous[road.to] = current
                    heappush(frontier, (candidate, road.to))
                    steps.append(self._snapshot(current, f"improve {road.to} to {candidate}", distances, frontier, settled))

        return distances, previous, steps

    def path_to(self, previous: dict[str, str | None], target: str) -> list[str]:
        path = []
        current: str | None = target

        while current is not None:
            path.append(current)
            current = previous[current]

        return path[::-1]

    def _snapshot(
        self,
        current: str,
        action: str,
        distances: dict[str, float],
        frontier: list[tuple[float, str]],
        settled: set[str],
    ) -> Step:
        return Step(
            current=current,
            action=action,
            distances=distances.copy(),
            frontier=sorted(frontier),
            settled=settled.copy(),
        )

## 3. Create a Toy City

Now make a map where each road has a cost. Think of the cost as minutes, coins, energy, or difficulty.

<details>
<summary>Hint: directed or two-way roads?</summary>

`two_way=True` adds the road in both directions. Leave it `False` if the road should behave like a one-way street.

</details>

In [3]:
city = MapGraph()
city.connect("Arcade", "Bakery", 4, two_way=True)
city.connect("Arcade", "Cinema", 2, two_way=True)
city.connect("Cinema", "Bakery", 1, two_way=True)
city.connect("Cinema", "Diner", 7, two_way=True)
city.connect("Bakery", "Diner", 3, two_way=True)
city.connect("Bakery", "Library", 6, two_way=True)
city.connect("Diner", "Library", 1, two_way=True)

for place, roads in city.roads.items():
    options = ", ".join(f"{road.to}({road.cost})" for road in roads)
    print(f"{place:>7} -> {options}")

 Arcade -> Bakery(4), Cinema(2)
 Bakery -> Arcade(4), Cinema(1), Diner(3), Library(6)
 Cinema -> Arcade(2), Bakery(1), Diner(7)
  Diner -> Cinema(7), Bakery(3), Library(1)
Library -> Bakery(6), Diner(1)


## 4. Run Dijkstra

Start at one place. The runner returns three things:

- `distances`: cheapest known cost to each place
- `previous`: breadcrumb links for rebuilding paths
- `steps`: snapshots for replaying the search

<details>
<summary>Quick check</summary>

The cheapest route to `Library` should avoid the expensive direct-looking options and go through the cheaper chain.

</details>

In [4]:
runner = DijkstraRunner(city)
distances, previous, steps = runner.shortest_paths("Arcade")

destination = "Library"
route = runner.path_to(previous, destination)

print("Shortest costs from Arcade:")
for place, cost in sorted(distances.items()):
    print(f"{place:>7}: {cost}")

print(f"\nBest route to {destination}:", " -> ".join(route))

Shortest costs from Arcade:
 Arcade: 0
 Bakery: 3
 Cinema: 2
  Diner: 6
Library: 7

Best route to Library: Arcade -> Cinema -> Bakery -> Diner -> Library


## 5. Replay the Search

The algorithm is easier to understand when you watch its state change.

The replay object below prints each snapshot: the current place, the action, the frontier, the settled places, and the current scoreboard.

<details>
<summary>Hint: what is relaxation?</summary>

Relaxation means: "I found a route to this neighbor. Is it cheaper than the best route I had before?" If yes, update the scoreboard.

</details>

**Trace model.** Define `SearchReplay`, the structure used to capture replayable algorithm state.


In [ ]:
class SearchReplay:
    def __init__(self, steps: list[Step]):
        self.steps = steps

    def show(self, limit: int | None = None) -> None:
        selected_steps = self.steps if limit is None else self.steps[:limit]

        for number, step in enumerate(selected_steps, start=1):
            print(f"Step {number}: {step.current} | {step.action}")
            print("  settled :", self._format_settled(step.settled))
            print("  frontier:", self._format_frontier(step.frontier))
            print("  scores  :", self._format_distances(step.distances))
            print()

    def _format_settled(self, settled: set[str]) -> str:
        return ", ".join(sorted(settled)) or "none"

    def _format_frontier(self, frontier: list[tuple[float, str]]) -> str:
        return ", ".join(f"{place}:{cost}" for cost, place in frontier) or "empty"

    def _format_distances(self, distances: dict[str, float]) -> str:
        parts = []
        for place, cost in sorted(distances.items()):
            label = "inf" if cost == inf else int(cost)
            parts.append(f"{place}={label}")
        return ", ".join(parts)


**Example state.** Create `replay`, the concrete values used in the next run.


In [ ]:
replay = SearchReplay(steps)

replay.show(limit=8)


## 6. Your Experiments

Try changing one thing at a time:

- Add a shortcut from `Arcade` to `Library`
- Make `Cinema` to `Diner` cheaper
- Add a new place called `Museum`
- Change the start or destination

<details>
<summary>Challenge</summary>

Predict the route before you run the cell. Then compare your guess with the output.

</details>

In [6]:
experiment = MapGraph()
experiment.connect("Arcade", "Bakery", 4, two_way=True)
experiment.connect("Arcade", "Cinema", 2, two_way=True)
experiment.connect("Cinema", "Bakery", 1, two_way=True)
experiment.connect("Cinema", "Diner", 5, two_way=True)
experiment.connect("Bakery", "Diner", 3, two_way=True)
experiment.connect("Diner", "Library", 1, two_way=True)
experiment.connect("Arcade", "Library", 9, two_way=True)
experiment.connect("Library", "Museum", 2, two_way=True)

experiment_runner = DijkstraRunner(experiment)
experiment_distances, experiment_previous, experiment_steps = experiment_runner.shortest_paths("Arcade")

for target in ["Library", "Museum"]:
    route = experiment_runner.path_to(experiment_previous, target)
    print(f"{target}: cost {experiment_distances[target]}, route {' -> '.join(route)}")

Library: cost 7, route Arcade -> Cinema -> Bakery -> Diner -> Library
Museum: cost 9, route Arcade -> Cinema -> Bakery -> Diner -> Library -> Museum


## Visual Trace + Rigor Studio

**Problem frame.** Find shortest paths from one source when all edge weights are nonnegative.

**Interactive animation target.** Animate settled nodes, frontier distances, and edge relaxations on the graph.

**Correctness handle.** Once a node is settled, no unsettled route can produce a shorter path to it.

**Complexity handle.** O((V + E) log V) with a heap; simpler teaching versions are O(V^2).

**Failure mode to test.** Negative edge weights break the settled-node invariant.

**Studio task.** Add a negative edge and explain why the visual trace becomes misleading.


In [ ]:
from pathlib import Path
import sys
from IPython.display import display

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "courseware").exists():
        sys.path.insert(0, str(candidate))
        break

from courseware import AlgorithmPlayer, dijkstra_trace, render_dijkstra_graph, render_trace_table

sample_graph = {
    "A": {"B": 2, "C": 5},
    "B": {"A": 2, "C": 1, "D": 4},
    "C": {"A": 5, "B": 1, "D": 1},
    "D": {"B": 4, "C": 1},
}
trace = dijkstra_trace(sample_graph, "A")
display(render_trace_table(trace))
AlgorithmPlayer(trace, render_dijkstra_graph).display()
